This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [27]:
testable_data = data.get_testable_data("Example\\inputs\\case study 2 input-open codes\\open no junk numerical.csv")
codes = data.get_codes("Example\\inputs\\case study 2 input-open codes\\hackathon open codes.csv")
all_scores = scores.get_MPNet_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
#make this go up to 88
all_scores_expanded[[str(i) for i in range(1, 89)]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Open code(s)" column of testable_data to all_scores_expanded
all_scores_expanded["Open code(s)"] = testable_data["Open code(s)"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

,Input phrase,1,2,3,4,5,6,7,8,9,...,80,81,82,83,84,85,86,87,88,Open code(s)
0,I don't know. I'm balancing ideas against prizes.,0.612352,0.510437,0.268802,0.237187,0.332806,0.192849,0.321369,0.366131,0.192078,...,0.145357,0.030819,0.154575,0.172357,0.296530,0.071397,0.244197,0.170808,0.204810,61
1,"Before you shoot them down, say them out loud.",-0.009469,-0.022985,0.022602,0.025652,0.014281,-0.023187,0.099532,0.061165,0.069228,...,-0.011638,0.129382,0.045929,0.163761,0.048185,0.088616,0.051341,0.039882,0.170376,39
2,But it's all blockchain related so.,0.066276,0.006575,-0.053198,0.105342,0.073612,0.042957,0.066776,0.108214,-0.052754,...,-0.044249,-0.029944,0.125121,0.039436,0.078353,0.079517,0.107389,0.034736,0.063548,43
3,Do you want blockchain?,0.154043,0.092580,0.064286,0.130277,0.070327,0.076795,0.122968,0.146893,0.047370,...,0.035324,0.001702,0.097759,0.047525,0.144195,0.149757,0.140559,0.085307,0.132995,43
4,"No, I, no, I know I'm thinking of it because I...",0.552655,0.429717,0.255136,0.252257,0.277053,0.176214,0.325410,0.293061,0.187409,...,-0.001295,-0.016985,0.169623,0.210769,0.357192,0.091249,0.223749,0.155566,0.219387,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228,Should I leave this here?,0.080211,0.097657,0.040056,0.158818,0.147333,0.135008,0.147233,0.179559,0.114704,...,0.022366,0.081857,0.125731,0.109896,0.135783,0.184340,0.189846,0.040759,0.114574,77
229,"Here we're all conquered and divided, right? Y...",-0.009710,0.034181,0.056213,0.083525,0.101431,0.101288,0.081463,0.056472,0.183759,...,-0.048761,0.061710,0.016801,0.166559,0.077694,0.098115,0.031495,0.266061,0.151248,26
230,"OK, like you gotta do like this spread. Leavin...",0.057108,-0.025628,0.002867,0.043828,0.088668,0.043008,0.076653,0.059193,0.121685,...,0.026359,0.060778,0.021174,0.054849,0.103873,0.136032,0.063165,0.144237,0.094586,77
231,When are you going to go to sleep?,-0.002549,0.013551,0.067133,0.041504,0.041088,0.066332,0.051286,0.063104,0.115495,...,-0.015630,0.034144,0.032536,0.041867,0.067029,0.084402,0.055292,0.244487,0.180621,87


In [28]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 10 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) >= min) 
        & (all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Open code(s)"].tolist()
    predictions = all_scores_expanded_filtered[[str(i) for i in range(1, 89)]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    #make these labels also go to 88 instead of 11
    f1s = f1_score(ground_truths, predictions, labels=list(range(1, 89)), average=None, zero_division=0.0) 
    #mtx = confusion_matrix(ground_truths, predictions, labels=list(range(1, 89)))
    kappa = cohen_kappa_score(ground_truths, predictions, weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    # count rows below the minimum threshold
    rows_below_min = len(all_scores_expanded[all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) < min])
    # also, count rows that had an "Open code(s)" of 0, that also had a max score below the minimum threshold
    rows_below_min_and_zero = len(all_scores_expanded[(all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) < min) & (all_scores_expanded["Open code(s)"] == 0)])
    return [f1, kappa, rows_below_min, rows_below_min_and_zero]

In [29]:
rows = []
total_rows = len(all_scores_expanded)
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows_below_min = results[2]
    percentage_below_min = (rows_below_min / total_rows) * 100
    rows_below_min_and_zero = results[3]
    percentage_below_min_and_zero = (rows_below_min_and_zero / total_rows) * 100
    rows.append({"min": i, "max": j, "kappa": results[1], "f1": results[0], "percentage_below_min": percentage_below_min, "rows_below_min": rows_below_min, "percentage_below_min_and_zero": percentage_below_min_and_zero, "rows_below_min_and_zero": rows_below_min_and_zero})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_MPNet_open_clean_hackathon.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeW

,min,max,kappa,f1,percentage_below_min,rows_below_min,percentage_below_min_and_zero,rows_below_min_and_zero
0,0.95,1.00,NaN,0.000000,100.000000,233,0.0,0
1,0.90,0.95,NaN,0.000000,100.000000,233,0.0,0
2,0.85,0.90,NaN,0.000000,100.000000,233,0.0,0
3,0.80,0.85,NaN,0.000000,100.000000,233,0.0,0
4,0.75,0.80,NaN,0.000000,100.000000,233,0.0,0
5,0.70,0.75,0.333333,0.011364,99.141631,231,0.0,0
6,0.65,0.70,0.000000,0.007576,98.283262,229,0.0,0
7,0.60,0.65,0.000000,0.000000,96.995708,226,0.0,0
8,0.55,0.60,0.166667,0.022727,92.703863,216,0.0,0
9,0.50,0.55,0.255319,0.022727,89.699571,209,0.0,0
